# Terraform IaC — Modules, State, Workspaces, DE Patterns

This notebook is a hands-on Terraform deep dive for a Senior Data Engineer workflow.

## Mental model

Terraform gives a DE team a repeatable way to create infrastructure as code:

- **Modules** package reusable infrastructure patterns.
- **State** is Terraform's memory of what exists.
- **Backends** make that state safe for teams.
- **Workspaces** isolate environments like dev and prod.
- **Provider patterns** let us stamp out data platform building blocks consistently.

## Fixed context used in this notebook

### PostgreSQL telemetry dataset
- Host: `localhost:5432`
- Database: `de_telemetry`
- User: `de_admin`
- Password: `DeAdmin2026!`

### Tables
- `endpoints`: 10,000 rows
- `metrics`: 500,000 rows
- `alerts`: 25,000 rows

### Narrative
A Citi-like DE environment monitors **6,000+ API endpoints** for latency, error rate, and throughput. Alerts escalate through severity tiers.

### Stack context
- Kafka: `localhost:9092`
- Spark: `pyspark==3.5.4`
- Airflow: `localhost:8082`
- MLflow: `localhost:5000`
- dbt: `C:/py_venv/proj_educate/Scripts/dbt.exe`
- Databricks host: `https://dbc-9f35a83d-b4e7.cloud.databricks.com`
- GCP project: `citi-de-learning`
- Azure subscription: `b3811436-61fc-4a3a-a6a9-deb05955076d`
- AWS profile: `study`
- AWS region: `us-east-1`
- AWS account: `357811130281`

> Design goal: use Terraform CLI against AWS profile `study`, create reusable Terraform code, inspect plans, demonstrate workspaces, and show a DE infrastructure pattern with S3 + Glue + Athena.


In [ ]:
from __future__ import annotations

import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path

AWS_PROFILE = "study"
AWS_DEFAULT_REGION = "us-east-1"
WINDOWS_WORKDIR = Path(r"D:/Workspace/Technologies/citi_terraform_advanced")

os.environ["AWS_PROFILE"] = AWS_PROFILE
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION
os.environ["PYTHONUTF8"] = "1"

def resolve_terraform() -> str:
    candidates = [
        r"C:\\Windows\\System32\\terraform.exe",
        shutil.which("terraform"),
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return str(candidate)
    raise FileNotFoundError(
        "Terraform executable not found. Expected C:\\Windows\\System32\\terraform.exe or terraform on PATH."
    )

def run_cmd(
    args,
    cwd: Path | None = None,
    check: bool = True,
    env: dict | None = None,
    echo: bool = True,
):
    if isinstance(args, str):
        display = args
        shell = True
    else:
        display = " ".join(shlex.quote(str(a)) for a in args)
        shell = False

    if echo:
        print(f"$ {display}")
        if cwd:
            print(f"cwd={cwd}")

    proc = subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        env=env or os.environ.copy(),
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        shell=shell,
    )

    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr, file=sys.stderr)

    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {display}")

    return proc

terraform_exe = resolve_terraform()
WINDOWS_WORKDIR.mkdir(parents=True, exist_ok=True)

print("Terraform executable:", terraform_exe)
print("Working directory:", WINDOWS_WORKDIR)
print("AWS_PROFILE:", os.environ["AWS_PROFILE"])
print("AWS_DEFAULT_REGION:", os.environ["AWS_DEFAULT_REGION"])

run_cmd([terraform_exe, "version"], cwd=WINDOWS_WORKDIR)


## 1) Setup

This section creates a clean Terraform project directory and writes provider scaffolding.

We intentionally keep:
- UTF-8-safe subprocess handling
- explicit `AWS_PROFILE=study`
- explicit `AWS_DEFAULT_REGION=us-east-1`

That makes local execution deterministic and easy to debug.


In [ ]:
module_dir = WINDOWS_WORKDIR / "modules" / "s3_lake"
module_dir.mkdir(parents=True, exist_ok=True)

providers_tf = textwrap.dedent("""
terraform {
  required_version = ">= 1.6.0"

  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

provider "aws" {
  profile = "study"
  region  = "us-east-1"
}
""").strip() + "\n"

(WINDOWS_WORKDIR / "providers.tf").write_text(providers_tf, encoding="utf-8")

print((WINDOWS_WORKDIR / "providers.tf").read_text(encoding="utf-8"))


## 2) Module Pattern

We now create a reusable module at:

`modules/s3_lake/`

It will contain:
- `main.tf`
- `variables.tf`
- `outputs.tf`

The module builds:
- an S3 bucket
- bucket versioning

Then the root module will call it twice:
- `dev`
- `prod`

This is the simplest real pattern a DE team can reuse for data lake storage.


In [ ]:
module_main_tf = textwrap.dedent("""
resource "aws_s3_bucket" "this" {
  bucket = var.bucket_name
  tags   = var.tags
}

resource "aws_s3_bucket_versioning" "this" {
  bucket = aws_s3_bucket.this.id

  versioning_configuration {
    status = "Enabled"
  }
}
""").strip() + "\n"

module_variables_tf = textwrap.dedent("""
variable "bucket_name" {
  description = "Globally unique S3 bucket name."
  type        = string
}

variable "tags" {
  description = "Tags applied to the bucket."
  type        = map(string)
  default     = {}
}
""").strip() + "\n"

module_outputs_tf = textwrap.dedent("""
output "bucket_arn" {
  value = aws_s3_bucket.this.arn
}

output "bucket_name" {
  value = aws_s3_bucket.this.bucket
}
""").strip() + "\n"

(module_dir / "main.tf").write_text(module_main_tf, encoding="utf-8")
(module_dir / "variables.tf").write_text(module_variables_tf, encoding="utf-8")
(module_dir / "outputs.tf").write_text(module_outputs_tf, encoding="utf-8")

for file_name in ["main.tf", "variables.tf", "outputs.tf"]:
    path = module_dir / file_name
    print(f"\n--- {path} ---")
    print(path.read_text(encoding="utf-8"))


In [ ]:
root_main_tf = textwrap.dedent("""
locals {
  base_tags = {
    owner       = "Sean Girgis"
    project     = "terraform_iac_basics"
    environment = terraform.workspace
    workload    = "data-engineering"
  }
}

module "s3_lake_dev" {
  source      = "./modules/s3_lake"
  bucket_name = "egirgis-citi-de-lake-dev-357811130281"
  tags = merge(local.base_tags, {
    env = "dev"
  })
}

module "s3_lake_prod" {
  source      = "./modules/s3_lake"
  bucket_name = "egirgis-citi-de-lake-prod-357811130281"
  tags = merge(local.base_tags, {
    env = "prod"
  })
}

output "dev_bucket_name" {
  value = module.s3_lake_dev.bucket_name
}

output "prod_bucket_name" {
  value = module.s3_lake_prod.bucket_name
}
""").strip() + "\n"

(WINDOWS_WORKDIR / "main.tf").write_text(root_main_tf, encoding="utf-8")

print((WINDOWS_WORKDIR / "main.tf").read_text(encoding="utf-8"))


In [ ]:
run_cmd([terraform_exe, "fmt", "-recursive"], cwd=WINDOWS_WORKDIR)
run_cmd([terraform_exe, "init", "-input=false"], cwd=WINDOWS_WORKDIR)

plan_proc = run_cmd(
    [terraform_exe, "plan", "-input=false", "-no-color"],
    cwd=WINDOWS_WORKDIR,
    check=False,
)

plan_text = (plan_proc.stdout or "") + "\n" + (plan_proc.stderr or "")
summary_match = re.search(r"Plan:\\s+(\\d+)\\s+to add,\\s+(\\d+)\\s+to change,\\s+(\\d+)\\s+to destroy\\.", plan_text)

if summary_match:
    adds, changes, destroys = summary_match.groups()
    print({"to_add": int(adds), "to_change": int(changes), "to_destroy": int(destroys)})
else:
    print("Plan summary pattern not found. Full output retained above for debugging.")


## 3) State Backend (S3)

Local state is dangerous for teams because:
- one engineer can drift from another
- state files can be overwritten
- secrets and sensitive IDs may end up in local files
- there is no safe locking by default

For teams, the common pattern is:
- **S3 backend** for shared state storage
- **DynamoDB lock table** for locking (older/common pattern)
- or a managed remote backend

Below we write the backend config exactly as a DE team would, but we do **not** re-init into S3 in this notebook because the backend bucket must already exist and be reachable.


In [ ]:
backend_tf = textwrap.dedent("""
terraform {
  backend "s3" {
    bucket = "egirgis-lab"
    key    = "citi-terraform/terraform.tfstate"
    region = "us-east-1"
    profile = "study"
  }
}
""").strip() + "\n"

backend_path = WINDOWS_WORKDIR / "backend_s3_example.tf"
backend_path.write_text(backend_tf, encoding="utf-8")

print(backend_path.read_text(encoding="utf-8"))
print("\nWhy we are not running `terraform init -migrate-state` here:")
print("- S3 backend bucket must already exist.")
print("- Team-safe remote state is a configuration pattern, not something we want to break mid-demo.")
print("- For a live migration, create the bucket first, then re-init with migration.")


## 4) Workspaces

Workspaces isolate **state**, not code.

That means:
- same Terraform code
- different state snapshots
- useful for `dev`, `prod`, sandbox-style separation

A common naming pattern is to include `${terraform.workspace}` in resource names.


In [ ]:
workspace_demo_tf = textwrap.dedent("""
resource "aws_s3_bucket" "workspace_named_demo" {
  bucket = "egirgis-ws-demo-${terraform.workspace}-357811130281"

  tags = {
    owner       = "Sean Girgis"
    environment = terraform.workspace
    purpose     = "workspace-demo"
  }
}
""").strip()

print(workspace_demo_tf)
print("\nThis snippet is illustrative and is not merged into main.tf to avoid bucket namespace collisions in the main demo.")


In [ ]:
existing_ws = run_cmd([terraform_exe, "workspace", "list"], cwd=WINDOWS_WORKDIR, check=False)

def ensure_workspace(name: str):
    ws_proc = run_cmd([terraform_exe, "workspace", "select", name], cwd=WINDOWS_WORKDIR, check=False)
    if ws_proc.returncode != 0:
        run_cmd([terraform_exe, "workspace", "new", name], cwd=WINDOWS_WORKDIR)

ensure_workspace("dev")
ensure_workspace("prod")

run_cmd([terraform_exe, "workspace", "list"], cwd=WINDOWS_WORKDIR)
run_cmd([terraform_exe, "workspace", "select", "default"], cwd=WINDOWS_WORKDIR)


## 5) DE Infrastructure Pattern

Now we build a more realistic DE module:

- S3 bucket as a small data lake landing zone
- Glue catalog database
- Athena workgroup

This is the kind of bundle a data engineering team provisions repeatedly for environment setup.

We will:
1. write a full module
2. call it from root
3. `terraform apply`
4. show outputs
5. `terraform destroy`

The names are fixed and tied to the AWS account context supplied in this notebook.


In [ ]:
de_module_dir = WINDOWS_WORKDIR / "modules" / "de_stack"
de_module_dir.mkdir(parents=True, exist_ok=True)

de_main_tf = textwrap.dedent("""
resource "aws_s3_bucket" "lake" {
  bucket = var.bucket_name
  tags   = var.tags
}

resource "aws_s3_bucket_versioning" "lake" {
  bucket = aws_s3_bucket.lake.id

  versioning_configuration {
    status = "Enabled"
  }
}

resource "aws_glue_catalog_database" "this" {
  name = var.glue_database_name
}

resource "aws_athena_workgroup" "this" {
  name = var.athena_workgroup_name

  configuration {
    enforce_workgroup_configuration    = true
    publish_cloudwatch_metrics_enabled = true

    result_configuration {
      output_location = "s3://${aws_s3_bucket.lake.bucket}/athena-results/"
    }
  }

  tags = var.tags
}
""").strip() + "\n"

de_variables_tf = textwrap.dedent("""
variable "bucket_name" {
  type = string
}

variable "glue_database_name" {
  type = string
}

variable "athena_workgroup_name" {
  type = string
}

variable "tags" {
  type    = map(string)
  default = {}
}
""").strip() + "\n"

de_outputs_tf = textwrap.dedent("""
output "bucket_name" {
  value = aws_s3_bucket.lake.bucket
}

output "bucket_arn" {
  value = aws_s3_bucket.lake.arn
}

output "glue_database_name" {
  value = aws_glue_catalog_database.this.name
}

output "athena_workgroup_name" {
  value = aws_athena_workgroup.this.name
}
""").strip() + "\n"

(de_module_dir / "main.tf").write_text(de_main_tf, encoding="utf-8")
(de_module_dir / "variables.tf").write_text(de_variables_tf, encoding="utf-8")
(de_module_dir / "outputs.tf").write_text(de_outputs_tf, encoding="utf-8")

for file_name in ["main.tf", "variables.tf", "outputs.tf"]:
    print(f"\n--- {de_module_dir / file_name} ---")
    print((de_module_dir / file_name).read_text(encoding="utf-8"))


In [ ]:
de_root_tf = textwrap.dedent("""
locals {
  de_tags = {
    owner       = "Sean Girgis"
    project     = "citi-de-stack"
    environment = "lab"
    dataset     = "de_telemetry"
  }
}

module "de_stack" {
  source                = "./modules/de_stack"
  bucket_name           = "egirgis-citi-de-stack-357811130281"
  glue_database_name    = "citi_de_telemetry"
  athena_workgroup_name = "citi_de_wg"
  tags                  = local.de_tags
}

output "de_bucket_name" {
  value = module.de_stack.bucket_name
}

output "de_bucket_arn" {
  value = module.de_stack.bucket_arn
}

output "de_glue_database_name" {
  value = module.de_stack.glue_database_name
}

output "de_athena_workgroup_name" {
  value = module.de_stack.athena_workgroup_name
}
""").strip() + "\n"

(WINDOWS_WORKDIR / "main.tf").write_text(de_root_tf, encoding="utf-8")

run_cmd([terraform_exe, "fmt", "-recursive"], cwd=WINDOWS_WORKDIR)
print((WINDOWS_WORKDIR / "main.tf").read_text(encoding="utf-8"))


In [ ]:
run_cmd([terraform_exe, "init", "-input=false"], cwd=WINDOWS_WORKDIR)

apply_proc = run_cmd(
    [terraform_exe, "apply", "-auto-approve", "-input=false", "-no-color"],
    cwd=WINDOWS_WORKDIR,
    check=False,
)

if apply_proc.returncode != 0:
    raise RuntimeError("terraform apply failed. Review the output above before continuing.")

outputs_proc = run_cmd(
    [terraform_exe, "output", "-json"],
    cwd=WINDOWS_WORKDIR,
)

outputs_json = json.loads(outputs_proc.stdout)
pretty_outputs = {k: v.get("value") for k, v in outputs_json.items()}
print(json.dumps(pretty_outputs, indent=2))


In [ ]:
destroy_proc = run_cmd(
    [terraform_exe, "destroy", "-auto-approve", "-input=false", "-no-color"],
    cwd=WINDOWS_WORKDIR,
    check=False,
)

if destroy_proc.returncode != 0:
    raise RuntimeError("terraform destroy failed. Review the output above before considering the demo complete.")


## 6) What Just Happened

**Modules** made infrastructure reusable.

**State backends** are what make Terraform safe for teams instead of just one laptop.

**Workspaces** let the same code isolate state for `dev` and `prod`.

**Provider patterns** let a DE team stamp out common infrastructure quickly and consistently.

### Citi-style takeaway

A data engineering team supporting thousands of monitored endpoints can use Terraform to provision:
- an S3 data lake landing zone
- a Glue catalog database
- an Athena workgroup

That turns repetitive setup into a reproducible workflow that can be created in minutes instead of by hand.

### Practical summary
- Use **modules** for repeatability
- Use **remote state** for team safety
- Use **workspaces** carefully for isolation
- Use Terraform to standardize DE platform primitives across environments
